In [1]:
!pip -q install requests pandas tqdm

In [2]:
import requests
import zipfile
import pandas as pd
import json
import time

from pathlib import Path
from tqdm.auto import tqdm

# ==========================================
# 1. CONFIG
# ==========================================
BASE_URL = (
    "https://transtats.bts.gov/PREZIP/"
    "On_Time_Reporting_Carrier_On_Time_Performance_1987_present"
)

OUTPUT_DIR = Path("/kaggle/working/BTS_ZIP")
TEMP_DIR = OUTPUT_DIR / "_temp"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)

START_YEAR, START_MONTH = 2022, 1
END_YEAR, END_MONTH = 2026, 3

CHUNK_SIZE = 200_000
MAX_RETRIES = 3

# ==========================================
# 2. DESIRED OUTPUT FIELDS
# ==========================================
FIELDS = [
    "FL_DATE", "YEAR", "QUARTER", "MONTH",
    "DAY_OF_MONTH", "DAY_OF_WEEK",
    "OP_CARRIER", "TAIL_NUM", "OP_CARRIER_FL_NUM",
    "ORIGIN_AIRPORT_ID", "DEST_AIRPORT_ID",
    "CRS_DEP_TIME", "DEP_TIME", "DEP_DELAY",
    "CRS_ARR_TIME", "ARR_TIME", "ARR_DELAY",
    "CANCELLED", "DIVERTED", "DISTANCE",
    "ORIGIN", "ORIGIN_CITY_NAME", "ORIGIN_STATE_ABR",
    "ORIGIN_STATE_FIPS", "ORIGIN_STATE_NM",
    "DEST", "DEST_CITY_NAME", "DEST_STATE_ABR",
    "DEST_STATE_FIPS", "DEST_STATE_NM",
    "OP_UNIQUE_CARRIER", "OP_CARRIER_AIRLINE_ID",
    "CARRIER_DELAY", "WEATHER_DELAY", "NAS_DELAY",
    "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY",
    "CANCELLATION_CODE"
]

# ==========================================
# 3. BTS SOURCE COLUMN ALIASES
#    target field -> possible source names
# ==========================================
COLUMN_ALIASES = {
    "FL_DATE": ["FL_DATE", "FlightDate"],
    "YEAR": ["YEAR", "Year"],
    "QUARTER": ["QUARTER", "Quarter"],
    "MONTH": ["MONTH", "Month"],
    "DAY_OF_MONTH": ["DAY_OF_MONTH", "DayofMonth"],
    "DAY_OF_WEEK": ["DAY_OF_WEEK", "DayOfWeek"],

    # Airline identifiers
    # IATA code is distinct from DOT unique carrier code.
    "OP_CARRIER": [
        "OP_CARRIER",
        "IATA_CODE_Reporting_Airline",
    ],
    "TAIL_NUM": ["TAIL_NUM", "Tail_Number"],
    "OP_CARRIER_FL_NUM": [
        "OP_CARRIER_FL_NUM",
        "Flight_Number_Reporting_Airline",
    ],
    "OP_UNIQUE_CARRIER": [
        "OP_UNIQUE_CARRIER",
        "Reporting_Airline",
    ],
    "OP_CARRIER_AIRLINE_ID": [
        "OP_CARRIER_AIRLINE_ID",
        "DOT_ID_Reporting_Airline",
    ],

    # Airport IDs
    "ORIGIN_AIRPORT_ID": ["ORIGIN_AIRPORT_ID", "OriginAirportID"],
    "DEST_AIRPORT_ID": ["DEST_AIRPORT_ID", "DestAirportID"],

    # Departure / arrival
    "CRS_DEP_TIME": ["CRS_DEP_TIME", "CRSDepTime"],
    "DEP_TIME": ["DEP_TIME", "DepTime"],
    "DEP_DELAY": ["DEP_DELAY", "DepDelay"],
    "CRS_ARR_TIME": ["CRS_ARR_TIME", "CRSArrTime"],
    "ARR_TIME": ["ARR_TIME", "ArrTime"],
    "ARR_DELAY": ["ARR_DELAY", "ArrDelay"],

    # Flight status
    "CANCELLED": ["CANCELLED", "Cancelled"],
    "DIVERTED": ["DIVERTED", "Diverted"],
    "DISTANCE": ["DISTANCE", "Distance"],
    "CANCELLATION_CODE": ["CANCELLATION_CODE", "CancellationCode"],

    # Origin
    "ORIGIN": ["ORIGIN", "Origin"],
    "ORIGIN_CITY_NAME": ["ORIGIN_CITY_NAME", "OriginCityName"],
    "ORIGIN_STATE_ABR": ["ORIGIN_STATE_ABR", "OriginState"],
    "ORIGIN_STATE_FIPS": ["ORIGIN_STATE_FIPS", "OriginStateFips"],
    "ORIGIN_STATE_NM": ["ORIGIN_STATE_NM", "OriginStateName"],

    # Destination
    "DEST": ["DEST", "Dest"],
    "DEST_CITY_NAME": ["DEST_CITY_NAME", "DestCityName"],
    "DEST_STATE_ABR": ["DEST_STATE_ABR", "DestState"],
    "DEST_STATE_FIPS": ["DEST_STATE_FIPS", "DestStateFips"],
    "DEST_STATE_NM": ["DEST_STATE_NM", "DestStateName"],

    # Delay causes
    "CARRIER_DELAY": ["CARRIER_DELAY", "CarrierDelay"],
    "WEATHER_DELAY": ["WEATHER_DELAY", "WeatherDelay"],
    "NAS_DELAY": ["NAS_DELAY", "NASDelay"],
    "SECURITY_DELAY": ["SECURITY_DELAY", "SecurityDelay"],
    "LATE_AIRCRAFT_DELAY": [
        "LATE_AIRCRAFT_DELAY",
        "LateAircraftDelay",
    ],
}

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (compatible; BTSDataPipeline/1.0)"
})


# ==========================================
# 4. GENERATE MONTHS
# ==========================================
def generate_months(sy, sm, ey, em):
    months = []
    year, month = sy, sm

    while (year, month) <= (ey, em):
        months.append((year, month))
        month += 1

        if month > 12:
            month = 1
            year += 1

    return months


# ==========================================
# 5. RESOLVE SOURCE COLUMNS
# ==========================================
def resolve_columns(source_columns):
    # Ignore leading/trailing whitespace and case differences.
    normalized = {
        str(col).strip().upper(): col
        for col in source_columns
    }

    mapping = {}

    for target in FIELDS:
        candidates = COLUMN_ALIASES.get(target, [target])

        for candidate in candidates:
            source = normalized.get(candidate.strip().upper())

            if source is not None:
                mapping[target] = source
                break

    return mapping


# ==========================================
# 6. VALIDATE EXISTING OUTPUT
# ==========================================
def is_valid_output(csv_path):
    if not csv_path.exists() or csv_path.stat().st_size == 0:
        return False

    try:
        header = pd.read_csv(csv_path, nrows=0).columns.tolist()

        if header != FIELDS:
            return False

        # Check that the file contains at least one data row.
        sample = pd.read_csv(csv_path, nrows=1)
        return not sample.empty

    except Exception:
        return False


# ==========================================
# 7. PROCESS ONE MONTH
# ==========================================
def process_month(year, month, retries=MAX_RETRIES):
    stem = f"On_Time_Reporting_{year}_{month}"
    zip_path = TEMP_DIR / f"{stem}.zip"
    csv_path = OUTPUT_DIR / f"{stem}.csv"
    temp_output = TEMP_DIR / f"{stem}_filtered.csv"

    # Skip only if the existing CSV passes basic validation.
    if is_valid_output(csv_path):
        return "SKIPPED"

    url = f"{BASE_URL}_{year}_{month}.zip"

    for attempt in range(1, retries + 1):
        try:
            # Remove incomplete files from a previous attempt.
            zip_path.unlink(missing_ok=True)
            temp_output.unlink(missing_ok=True)

            # --------------------------------------
            # A. Download monthly ZIP
            # --------------------------------------
            with session.get(
                url,
                stream=True,
                timeout=(30, 180),
            ) as response:

                if response.status_code == 404:
                    return "NOT_AVAILABLE"

                response.raise_for_status()

                with open(zip_path, "wb") as f:
                    for chunk in response.iter_content(
                        chunk_size=1024 * 1024
                    ):
                        if chunk:
                            f.write(chunk)

            if not zipfile.is_zipfile(zip_path):
                raise ValueError(
                    "Downloaded response is not a valid ZIP file"
                )

            # --------------------------------------
            # B. Inspect ZIP and read source header
            # --------------------------------------
            with zipfile.ZipFile(zip_path) as archive:
                csv_members = [
                    name for name in archive.namelist()
                    if name.lower().endswith(".csv")
                ]

                if not csv_members:
                    raise ValueError("No CSV file found inside ZIP")

                source_name = csv_members[0]

                with archive.open(source_name) as source:
                    source_header = pd.read_csv(
                        source,
                        nrows=0,
                        encoding="utf-8-sig",
                    ).columns.tolist()

                mapping = resolve_columns(source_header)

                missing = [
                    field for field in FIELDS
                    if field not in mapping
                ]

                print(
                    f"{year}-{month:02d}: "
                    f"{len(mapping)}/{len(FIELDS)} fields matched"
                )

                if missing:
                    raise ValueError(
                        "Unmapped fields: " + ", ".join(missing)
                    )

                # --------------------------------------
                # C. Read and filter in chunks
                # --------------------------------------
                source_columns = list(mapping.values())

                with archive.open(source_name) as source:
                    reader = pd.read_csv(
                        source,
                        usecols=source_columns,
                        chunksize=CHUNK_SIZE,
                        encoding="utf-8-sig",
                        low_memory=False,
                    )

                    first_chunk = True
                    total_rows = 0

                    reverse_mapping = {
                        source_col: target
                        for target, source_col in mapping.items()
                    }

                    for chunk in reader:
                        chunk.rename(
                            columns=reverse_mapping,
                            inplace=True,
                        )

                        chunk = chunk[FIELDS]

                        chunk.to_csv(
                            temp_output,
                            mode="w" if first_chunk else "a",
                            index=False,
                            header=first_chunk,
                        )

                        total_rows += len(chunk)
                        first_chunk = False

                if total_rows == 0:
                    raise ValueError(
                        "Source CSV contains no data rows"
                    )

            # --------------------------------------
            # D. Validate then commit output
            # --------------------------------------
            if not temp_output.exists():
                raise ValueError("Filtered CSV was not generated")

            if temp_output.stat().st_size == 0:
                raise ValueError("Filtered CSV is empty")

            output_header = pd.read_csv(
                temp_output,
                nrows=0,
            ).columns.tolist()

            if output_header != FIELDS:
                raise ValueError(
                    "Output column order does not match FIELDS"
                )

            # Replace the final CSV only after processing succeeds.
            temp_output.replace(csv_path)

            print(
                f"  Rows: {total_rows:,} | "
                f"Output size: {csv_path.stat().st_size / (1024**2):.2f} MB"
            )

            return "SUCCESS"

        except Exception as error:
            print(
                f"Attempt {attempt}/{retries} "
                f"for {year}-{month:02d}: {error}"
            )

            zip_path.unlink(missing_ok=True)
            temp_output.unlink(missing_ok=True)

            if attempt < retries:
                time.sleep(attempt * 2)

        finally:
            # ZIPs are temporary; retain only filtered CSV outputs.
            zip_path.unlink(missing_ok=True)

    return "FAILED"


# ==========================================
# 8. RUN ALL MONTHS
# ==========================================
results = {}

months = generate_months(
    START_YEAR, START_MONTH,
    END_YEAR, END_MONTH,
)

for year, month in tqdm(months, desc="BTS monthly processing"):
    key = f"{year}-{month:02d}"
    status = process_month(year, month)

    results[key] = status
    print(f"{key}: {status}")


# ==========================================
# 9. SAVE MANIFEST
# ==========================================
manifest = {
    "source": "BTS TranStats",
    "period": (
        f"{START_YEAR}-{START_MONTH:02d} to "
        f"{END_YEAR}-{END_MONTH:02d}"
    ),
    "expected_months": len(months),
    "selected_fields": FIELDS,
    "results": results,
    "generated_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
}

manifest_path = OUTPUT_DIR / "manifest.json"

manifest_path.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

# ==========================================
# 10. SUMMARY
# ==========================================
print("\n========== SUMMARY ==========")

for status in [
    "SUCCESS",
    "SKIPPED",
    "NOT_AVAILABLE",
    "FAILED",
]:
    count = sum(value == status for value in results.values())
    print(f"{status}: {count}")

print("Expected months:", len(months))
print("CSV count:", len(list(OUTPUT_DIR.glob("*.csv"))))
print("Manifest:", manifest_path)
print("Output directory:", OUTPUT_DIR)

BTS monthly processing:   0%|          | 0/51 [00:00<?, ?it/s]

2022-01: 38/38 fields matched
  Rows: 537,902 | Output size: 95.31 MB
2022-01: SUCCESS
2022-02: 38/38 fields matched
  Rows: 495,713 | Output size: 88.01 MB
2022-02: SUCCESS
2022-03: 38/38 fields matched
  Rows: 564,853 | Output size: 100.75 MB
2022-03: SUCCESS
2022-04: 38/38 fields matched
  Rows: 556,502 | Output size: 99.23 MB
2022-04: SUCCESS
2022-05: 38/38 fields matched
  Rows: 578,819 | Output size: 103.15 MB
2022-05: SUCCESS
2022-06: 38/38 fields matched
  Rows: 577,283 | Output size: 103.00 MB
2022-06: SUCCESS
2022-07: 38/38 fields matched
  Rows: 594,957 | Output size: 106.30 MB
2022-07: SUCCESS
2022-08: 38/38 fields matched
  Rows: 589,810 | Output size: 105.18 MB
2022-08: SUCCESS
2022-09: 38/38 fields matched
  Rows: 557,494 | Output size: 99.01 MB
2022-09: SUCCESS
2022-10: 38/38 fields matched
  Rows: 572,287 | Output size: 102.19 MB
2022-10: SUCCESS
2022-11: 38/38 fields matched
  Rows: 546,410 | Output size: 97.75 MB
2022-11: SUCCESS
2022-12: 38/38 fields matched
  Rows: